Connect to Aiven PostgreSQL using SQLAlchemy and print the PostgreSQL version

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env")

HOST = os.getenv("AIVEN_HOST")
PORT = os.getenv("AIVEN_PORT")
DB = os.getenv("AIVEN_DB")
USER = os.getenv("AIVEN_USER")
PASSWORD = os.getenv("AIVEN_PASSWORD")
SSLMODE = os.getenv("AIVEN_SSLMODE", "require")

DATABASE_URL = (
    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}"
    f"?sslmode={SSLMODE}"
)

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone()[0])

PostgreSQL 17.10 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.1 20260123 (Red Hat 15.2.1-7), 64-bit


Load sample data into the database

In [2]:
# Check database state
from sqlalchemy import text

query = """
SELECT table_schema,
       table_name
FROM information_schema.tables
WHERE table_schema NOT IN (
    'information_schema',
    'pg_catalog'
)
ORDER BY table_schema, table_name;
"""

with engine.connect() as conn:
    rows = conn.execute(text(query)).fetchall()

print(f"Tables found: {len(rows)}")

for row in rows[:20]:
    print(row)
    
# Extensions
with engine.begin() as conn:
    conn.execute(text('CREATE EXTENSION IF NOT EXISTS tablefunc;'))
    conn.execute(text('CREATE EXTENSION IF NOT EXISTS "uuid-ossp";'))

print("Extensions enabled.")

Tables found: 0
Extensions enabled.


In [4]:
import os
import subprocess
from dotenv import load_dotenv

load_dotenv("../.env")

DATABASE_URL = os.getenv("AIVEN_DATABASE_URL")

PG_RESTORE = r"D:\Apps\pgAdmin 4\runtime\pg_restore.exe"
DUMP_FILE = "AdventureWorksPG.gz"

cmd = [
    PG_RESTORE,
    "--verbose",
    "--no-owner",
    "--no-privileges",
    "--clean",
    "--if-exists",
    "--dbname",
    DATABASE_URL,
    DUMP_FILE
]

print("Starting restore...")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("Return Code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-5000:])

print("\nSTDERR:")
print(result.stderr[-5000:])

Starting restore...
Return Code: 1

STDOUT:


STDERR:
korderrouting FK_WorkOrderRouting_Location_LocationID"
pg_restore: creating FK CONSTRAINT "production.workorderrouting FK_WorkOrderRouting_WorkOrder_WorkOrderID"
pg_restore: creating FK CONSTRAINT "production.workorder FK_WorkOrder_Product_ProductID"
pg_restore: creating FK CONSTRAINT "production.workorder FK_WorkOrder_ScrapReason_ScrapReasonID"
pg_restore: creating FK CONSTRAINT "purchasing.productvendor FK_ProductVendor_Product_ProductID"
pg_restore: creating FK CONSTRAINT "purchasing.productvendor FK_ProductVendor_UnitMeasure_UnitMeasureCode"
pg_restore: creating FK CONSTRAINT "purchasing.productvendor FK_ProductVendor_Vendor_BusinessEntityID"
pg_restore: creating FK CONSTRAINT "purchasing.purchaseorderdetail FK_PurchaseOrderDetail_Product_ProductID"
pg_restore: creating FK CONSTRAINT "purchasing.purchaseorderdetail FK_PurchaseOrderDetail_PurchaseOrderHeader_PurchaseOrderID"
pg_restore: creating FK CONSTRAINT "purchasing.purchase

In [6]:
# Perform post-import checks

from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(text("""
        SELECT table_schema, COUNT(*) AS tables
        FROM information_schema.tables
        WHERE table_type = 'BASE TABLE'
        AND table_schema NOT IN ('pg_catalog', 'information_schema')
        GROUP BY table_schema
        ORDER BY tables DESC;
    """)).fetchall()

for row in rows:
    print(row)
    
with engine.connect() as conn:
    count = conn.execute(text("""
        SELECT COUNT(*) 
        FROM sales.salesorderheader;
    """)).scalar()

print(count)

('production', 25)
('sales', 19)
('person', 13)
('humanresources', 6)
('purchasing', 5)
31465


Schema Extraction

In [7]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect, text

load_dotenv(".env", override=True)

DATABASE_URL = os.getenv("AIVEN_DATABASE_URL")

if DATABASE_URL.startswith("postgres://"):
    DATABASE_URL = DATABASE_URL.replace(
        "postgres://",
        "postgresql+psycopg2://",
        1
    )

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())

defaultdb


In [9]:
inspector = inspect(engine)

TARGET_SCHEMAS = [
    "sales",
    "production",
    "person",
    "purchasing",
    "humanresources",
]

metadata = []

for schema in TARGET_SCHEMAS:
    for table in inspector.get_table_names(schema=schema):
        columns = inspector.get_columns(table, schema=schema)
        primary_key = inspector.get_pk_constraint(table, schema=schema)
        foreign_keys = inspector.get_foreign_keys(table, schema=schema)

        metadata.append({
            "schema": schema,
            "table": table,
            "full_table_name": f"{schema}.{table}",
            "columns": [
                {
                    "name": col["name"],
                    "type": str(col["type"]),
                    "nullable": col["nullable"],
                    "default": str(col.get("default")) if col.get("default") else None,
                }
                for col in columns
            ],
            "primary_key": primary_key.get("constrained_columns", []),
            "foreign_keys": [
                {
                    "columns": fk.get("constrained_columns", []),
                    "referred_schema": fk.get("referred_schema"),
                    "referred_table": fk.get("referred_table"),
                    "referred_columns": fk.get("referred_columns", []),
                }
                for fk in foreign_keys
            ],
        })

print("Tables extracted:", len(metadata))

output_path = Path("backend/schema_metadata.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved to:", output_path)

C:\Users\adars\AppData\Local\Temp\ipykernel_21536\1756978830.py:15: SAWarning: Did not recognize type 'xml' of column 'demographics'
  columns = inspector.get_columns(table, schema=schema)
C:\Users\adars\AppData\Local\Temp\ipykernel_21536\1756978830.py:15: SAWarning: Did not recognize type 'xml' of column 'catalogdescription'
  columns = inspector.get_columns(table, schema=schema)
C:\Users\adars\AppData\Local\Temp\ipykernel_21536\1756978830.py:15: SAWarning: Did not recognize type 'xml' of column 'instructions'
  columns = inspector.get_columns(table, schema=schema)
C:\Users\adars\AppData\Local\Temp\ipykernel_21536\1756978830.py:15: SAWarning: Did not recognize type 'xml' of column 'diagram'
  columns = inspector.get_columns(table, schema=schema)
C:\Users\adars\AppData\Local\Temp\ipykernel_21536\1756978830.py:15: SAWarning: Did not recognize type 'xml' of column 'additionalcontactinfo'
  columns = inspector.get_columns(table, schema=schema)
C:\Users\adars\AppData\Local\Temp\ipykernel_2

Tables extracted: 68
Saved to: backend\schema_metadata.json


Build schema docs

In [10]:
# Load metadata

import json

with open("backend/schema_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(len(metadata))

68


In [11]:
schema_docs = []

for table in metadata:

    columns = [
        c["name"]
        for c in table["columns"]
    ]

    fk_strings = []

    for fk in table["foreign_keys"]:
        fk_strings.append(
            f"{fk['columns']} -> "
            f"{fk['referred_schema']}."
            f"{fk['referred_table']}."
            f"{fk['referred_columns']}"
        )

    document = f"""
Table: {table['full_table_name']}

Columns:
{", ".join(columns)}

Primary Key:
{", ".join(table['primary_key'])}

Relationships:
{chr(10).join(fk_strings)}
"""

    schema_docs.append({
        "table": table["full_table_name"],
        "document": document.strip()
    })

print(schema_docs[0])

{'table': 'sales.countryregioncurrency', 'document': "Table: sales.countryregioncurrency\n\nColumns:\ncountryregioncode, currencycode, modifieddate\n\nPrimary Key:\ncountryregioncode, currencycode\n\nRelationships:\n['countryregioncode'] -> person.countryregion.['countryregioncode']\n['currencycode'] -> sales.currency.['currencycode']"}


In [12]:
with open(
    "backend/schema_documents.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        schema_docs,
        f,
        indent=2
    )

print("saved")

saved
